# Worm outlines in napari and spot density per worm area
Draw one outline per field around the worm to analyse, then count spots inside it and normalise by worm area.

Run this notebook in the **wormfish_annotation** environment (napari).

## Imports and setup

In [ ]:
import os
import sys
from pathlib import Path

# Make the conda environment's DLLs findable even if VS Code could not activate the environment
# (otherwise the kernel crashes when numpy is imported)
env_dir = Path(sys.executable).parent
for d in [env_dir, env_dir / "Library" / "mingw-w64" / "bin", env_dir / "Library" / "usr" / "bin",
          env_dir / "Library" / "bin", env_dir / "Scripts"]:
    if d.exists():
        os.environ["PATH"] = str(d) + os.pathsep + os.environ.get("PATH", "")
        os.add_dll_directory(str(d))

# Add the DotDetection folder to Python path
project_root = Path.cwd().parent  # Assuming notebook is in Notebooks folder
module_path = project_root / "DotDetection"
sys.path.insert(0, str(module_path))

# Import
from WormOutline_functions import prepare_outline_inputs, check_outline_status, annotate_outlines, summarize_outlines

In [ ]:
import os

raw_root = r"E:\David\TiffConverted"                                  # Raw TIFF stacks
pre_root = r"E:\David\TiffConverted_Preprocessed_HPC"                  # Preprocessed stacks (hot pixel lists)
dots_root = os.path.join(pre_root, "DetectedDots_A647thr100_CY3Bthr200_a0.7_b2.5_trim4")  # Spot detection results
inputs_root = os.path.join(pre_root, "WormOutlines", "inputs")            # MIPs + spots for napari
outlines_root = os.path.join(pre_root, "WormOutlines", "outlines")        # Drawn outlines

## Prepare napari inputs (once)
Saves per field a DAPI / A647 / CY3B MIP of the analysed planes (hot pixels corrected) and the detected spot positions.
Fields that are already prepared are skipped.

In [ ]:
prepare_outline_inputs(raw_root, pre_root, dots_root, inputs_root, z_trim=4)

## Check outline status

In [ ]:
check_outline_status(inputs_root, outlines_root)

## Draw worm outlines
A napari window opens with the first field that has no outline yet.

1. DAPI (gray) and A647 (magenta) MIPs are shown; tick CY3B or the spot layers in the layer list if needed
2. The 'worm outline' layer is active in polygon mode: click around the worm, double-click to close the polygon
   (several polygons are merged into one outline; use the shape tools to edit or delete)
3. Press `s` to save and load the next field

**Hotkeys:**
- `s`: Save outline and load next field
- `n`: No usable worm in this field (e.g. only embryos) and load next field
- `q`: Quit (run the cell again later to continue with the remaining fields)

Set `redo=True` to revisit fields that already have an outline (the saved polygon is loaded for editing).

In [ ]:
annotate_outlines(inputs_root, outlines_root, redo=False)

## Redo specific fields
Re-open fields by file name to fix a mistake. A saved outline is loaded for editing; saving replaces a 'no usable worm' mark,
and pressing `n` replaces an outline.

In [ ]:
annotate_outlines(inputs_root, outlines_root, only=["ProtK_Male head"])

## Spot density per worm area
Spots inside each outline (detected and decomposed, both channels), worm area (µm²) and spots per 1000 µm².
Saved as `worm_outline_counts.csv` in the outlines folder.

In [ ]:
rows = summarize_outlines(inputs_root, outlines_root, dots_root)